# Unsupervised Isolation Forest "Challenger" Model for Financial Fraud Detection

This notebook implements the training pipeline for an **Isolation Forest** model to act as an unsupervised **Challenger** running in **Shadow Mode** alongside the XGBoost **Champion** model.

### Key Objectives:
1. **Strict Unsupervised Training**: Preprocess the banking data and isolate/drop the target `is_fraud` label prior to training, ensuring the model identifies anomalies blindly.
2. **Contamination Estimation**: Dynamically estimate the contamination parameter based on the known fraud ratio in the historical dataset.
3. **Risk Score Transformation**: Convert the raw negative anomaly scores returned by the Isolation Forest's `decision_function()` to a standardized `0 - 100` Risk Score scale (100 = Highest Fraud Risk) matching the XGBoost champion's score range.
4. **Offline Evaluation**: Evaluate the unsupervised model's performance against the ground truth labels across various risk thresholds.
5. **Model Export**: Save the trained Isolation Forest model as a joblib file for downstream deployment in Flink.

## 1. Environment Setup

Installs the required Python packages: `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `seaborn`, and `joblib`.

In [ ]:
!pip install -q scikit-learn==1.6.1 pandas==2.2.3 numpy==1.26.4 matplotlib==3.10.0 seaborn==0.13.2 joblib==1.4.2

## 2. Imports and Configuration

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
)

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

## 3. Data Loading

We load the dataset `ml_ready_data_csv` generated by the feature pipeline. This is the exact same data source fed into the XGBoost model.
We set up a robust path resolution to automatically find the CSV file in local environments or Google Colab.

In [ ]:
# --- Data Path Configuration ---
# Supports running in local IDEs (from project root or ml_training folder) and Google Colab
DATA_PATHS = [
    Path("/content/ml_fraud_features.csv"),
    Path("../data_generation/ml_ready_data_csv/part-00000-279e5417-b24d-44e8-a7e9-7e7555b23b59-c000.csv"),
    Path("data_generation/ml_ready_data_csv/part-00000-279e5417-b24d-44e8-a7e9-7e7555b23b59-c000.csv"),
]

DATA_PATH = None
for p in DATA_PATHS:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    DATA_PATH = Path("/content/ml_fraud_features.csv")
    if not DATA_PATH.exists():
        try:
            from google.colab import files
            print("CSV not found. Please upload the data file...")
            uploaded = files.upload()
            uploaded_name = next(iter(uploaded))
            DATA_PATH = Path("/content") / uploaded_name
        except ImportError:
            raise FileNotFoundError(
                f"Data CSV file not found. Checked: {[str(p) for p in DATA_PATHS]}"
            )

print(f"Loading data from: {DATA_PATH.resolve()}")
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
display(df_raw.head())

## 4. Feature Preprocessing

We implement the exact same preprocessor logic as the XGBoost model to prepare our features:
- **`ip_address`**: Frequency Encoding (representing the frequency of each IP).
- **`event_type`**: One-Hot Encoding (converting events to binary indicator columns).
- **Numerical columns**: `amount`, `tx_count_1h`, `amount_avg_1h`, `amount_vs_avg`, `hour_of_day`.

In [ ]:
FEATURE_COLS = [
    "event_type",
    "amount",
    "ip_address",
    "tx_count_1h",
    "amount_avg_1h",
    "amount_vs_avg",
    "hour_of_day",
]
TARGET_COL = "is_fraud"
NUMERIC_COLS = ["amount", "tx_count_1h", "amount_avg_1h", "amount_vs_avg", "hour_of_day"]

@dataclass
class FraudFeaturePreprocessor:
    """Matches the preprocessing of the XGBoost Champion model."""
    event_type_categories: list[str] | None = None
    ip_frequency_map: dict[str, float] | None = None

    def fit(self, frame: pd.DataFrame) -> FraudFeaturePreprocessor:
        self.event_type_categories = sorted(frame["event_type"].astype(str).unique().tolist())
        ip_counts = frame["ip_address"].astype(str).value_counts()
        self.ip_frequency_map = ip_counts.to_dict()
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        if self.event_type_categories is None or self.ip_frequency_map is None:
            raise RuntimeError("Run fit() before transform().")

        working = frame.copy()

        # Frequency encoding for IP
        working["ip_address_freq"] = (
            working["ip_address"]
            .astype(str)
            .map(self.ip_frequency_map)
            .fillna(0)
            .astype(np.float32)
        )

        # One-Hot Encoding for event_type
        event_ohe = pd.get_dummies(
            working["event_type"].astype(str),
            prefix="event_type",
            dtype=np.float32,
        )
        for category in self.event_type_categories:
            col_name = f"event_type_{category}"
            if col_name not in event_ohe.columns:
                event_ohe[col_name] = 0.0
        event_ohe = event_ohe[[f"event_type_{c}" for c in self.event_type_categories]]

        numeric_part = working[NUMERIC_COLS].astype(np.float32)
        ip_part = working[["ip_address_freq"]]

        features = pd.concat([numeric_part, ip_part, event_ohe], axis=1)
        features = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return features

    def feature_names(self) -> list[str]:
        if self.event_type_categories is None:
            raise RuntimeError("Preprocessor has not been fit.")
        return NUMERIC_COLS + ["ip_address_freq"] + [
            f"event_type_{c}" for c in self.event_type_categories
        ]

# Fit and transform dataset
preprocessor = FraudFeaturePreprocessor().fit(df_raw[FEATURE_COLS])
df_features = preprocessor.transform(df_raw[FEATURE_COLS])

# Re-assemble preprocessed dataset 'df' containing features + target
df = df_features.copy()
df[TARGET_COL] = df_raw[TARGET_COL].astype(int).values

print(f"Preprocessed dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Feature columns ({len(preprocessor.feature_names())}): {preprocessor.feature_names()}")
display(df.head())

## 5. Unsupervised Setup & Data Isolation

As an unsupervised model, the Isolation Forest must be trained without seeing the labels.
We explicitly drop the `is_fraud` column from our training data and isolate the labels `y_eval` strictly for post-training validation.

In [ ]:
# CRITICAL: Isolate target label for evaluation purposes ONLY
y_eval = df[TARGET_COL].values

# Drop the is_fraud column from the training dataframe
X_train = df.drop(columns=[TARGET_COL])

print(f"X_train features shape: {X_train.shape}")
print(f"X_train columns: {list(X_train.columns)}")
assert TARGET_COL not in X_train.columns, "CRITICAL ERROR: is_fraud label leaked into training data!"

## 6. Isolation Forest Training

The `contamination` parameter defines the proportion of outliers in the data. We estimate this parameter using the historical proportion of fraud in our dataset (ratio of 1s in `is_fraud`).

In [ ]:
# Estimate the contamination parameter based on the known proportion of fraud
contamination = float(np.mean(y_eval))
print(f"Estimated contamination parameter (historical fraud ratio): {contamination:.6f} (~{contamination*100:.2f}%)")

# Initialize the Isolation Forest classifier
# Set random_state=RANDOM_STATE for reproducibility
iso_forest = IsolationForest(
    n_estimators=150,
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Training Isolation Forest model on X_train...")
iso_forest.fit(X_train)
print("Training completed!")

## 7. Scoring & Risk Mapping (Shadow Mode Logic)

Isolation Forest's `decision_function()` returns anomaly scores where:
- Outliers (anomalies) have **negative** scores.
- Inliers (normal events) have **positive** scores.

To deploy this challenger alongside our XGBoost champion, we must transform these scores to a **0 to 100 Risk Score scale**, where **100 represents the highest fraud risk**.
We implement a Min-Max scaling on the inverted anomaly scores:
$$RiskScore = 100 \times \frac{s_{\max} - s}{s_{\max} - s_{\min}}$$

For production/streaming inference in Flink, we save the minimum and maximum raw scores from the training set, allowing us to map incoming streaming events consistently.

In [ ]:
# Get raw anomaly scores
raw_scores = iso_forest.decision_function(X_train)

# Identify the min and max scores in our training set
raw_min = float(raw_scores.min())
raw_max = float(raw_scores.max())
print(f"Raw anomaly scores range: [{raw_min:.5f}, {raw_max:.5f}]")

# Map raw scores to [0, 100] Risk Score scale
# Lower raw_score = more anomalous = higher risk score
risk_scores = 100.0 * (raw_max - raw_scores) / (raw_max - raw_min)

# Save risk scores in our dataframe
df["risk_score"] = risk_scores

print("\nSample of Risk Scores vs. Ground Truth:")
display(df[[TARGET_COL, "risk_score"]].head(10))

# Plot the distribution of Risk Scores for Fraud vs. Legit
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x="risk_score", hue=TARGET_COL, kde=True, bins=50, multiple="stack")
plt.title("Distribution of Risk Scores by Class (0 = Legit, 1 = Fraud)")
plt.xlabel("Risk Score (0 - 100)")
plt.ylabel("Count")
plt.show()

## 8. Evaluation Against Ground Truth

We convert continuous risk scores into binary predictions using a defined risk threshold (e.g., Risk Score > 90) and evaluate using Precision, Recall, and F1-Score.

In [ ]:
# Define threshold to convert scores to binary predictions
threshold = 90.0
print(f"Evaluating model at Risk Score threshold: > {threshold}")

y_pred = (df["risk_score"] > threshold).astype(int)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_eval, y_pred, digits=4))

# Compute metrics
prec = precision_score(y_eval, y_pred)
rec = recall_score(y_eval, y_pred)
f1 = f1_score(y_eval, y_pred)

print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Plot Confusion Matrix
cm = confusion_matrix(y_eval, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Fraud"]).plot(
    cmap="Oranges", ax=ax, values_format=","
)
ax.set_title(f"Confusion Matrix (Threshold > {threshold})")
plt.tight_layout()
plt.show()

### 8.1 Threshold Tuning

Let's evaluate how Precision, Recall, and F1-Score vary across different thresholds. This will help find the optimal threshold for different business objectives (e.g., high recall to catch all fraud vs. high precision to avoid false alarms).

In [ ]:
thresholds = np.linspace(0, 100, 101)
precisions = []
recalls = []
f1_scores = []

for t in thresholds:
    pred = (df["risk_score"] > t).astype(int)
    precisions.append(precision_score(y_eval, pred, zero_division=0))
    recalls.append(recall_score(y_eval, pred, zero_division=0))
    f1_scores.append(f1_score(y_eval, pred, zero_division=0))

# Plot metrics vs. Threshold
plt.figure(figsize=(12, 6))
plt.plot(thresholds, precisions, label="Precision", color="blue", lw=2)
plt.plot(thresholds, recalls, label="Recall", color="green", lw=2)
plt.plot(thresholds, f1_scores, label="F1-Score", color="red", lw=2)

# Highlight best F1 threshold
best_idx = np.argmax(f1_scores)
best_t = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
plt.axvline(best_t, color="gray", linestyle="--", label=f"Best F1 Threshold ({best_t:.1f})")

plt.title("Model Metrics vs. Risk Score Threshold")
plt.xlabel("Risk Score Threshold")
plt.ylabel("Score")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

print(f"Optimal F1 threshold: {best_t:.1f} with F1-Score: {best_f1:.4f}")

## 9. Model Export

We save the trained Isolation Forest model as `isolation_forest_model.joblib`. We also export the associated metadata (preprocessor artifact, and raw score min/max ranges for risk score normalization) to ensure standard reproducibility.

In [ ]:
MODEL_EXPORT_PATH = Path("isolation_forest_model.joblib")
METADATA_EXPORT_PATH = Path("isolation_forest_metadata.joblib")

# Save model
joblib.dump(iso_forest, MODEL_EXPORT_PATH)

# Save metadata
metadata = {
    "raw_min": raw_min,
    "raw_max": raw_max,
    "contamination": contamination,
    "feature_names": preprocessor.feature_names(),
    "preprocessor_artifact": preprocessor.to_artifact(),
}
joblib.dump(metadata, METADATA_EXPORT_PATH)

print(f"Saved Isolation Forest model: {MODEL_EXPORT_PATH.resolve()} ({MODEL_EXPORT_PATH.stat().st_size / 1024:.1f} KB)")
print(f"Saved model metadata: {METADATA_EXPORT_PATH.resolve()} ({METADATA_EXPORT_PATH.stat().st_size / 1024:.1f} KB)")

## 10. Shadow Mode & Flink Integration Guide

For running the Challenger model in PyFlink/Python Shadow Mode alongside the XGBoost Champion, implement the following steps:

1. **Load Artifacts**:
   ```python
   import joblib
   import numpy as np

   model = joblib.load("isolation_forest_model.joblib")
   metadata = joblib.load("isolation_forest_metadata.joblib")
   raw_min = metadata["raw_min"]
   raw_max = metadata["raw_max"]
   ```

2. **Inference Function**:
   ```python
   def predict_risk_score(feature_vector: np.ndarray) -> float:
       # Get raw anomaly score from Isolation Forest
       # Reshape for single sample
       raw_score = model.decision_function(feature_vector.reshape(1, -1))[0]
       
       # Normalize to [0, 100]
       risk_score = 100.0 * (raw_max - raw_score) / (raw_max - raw_min)
       
       # Clip to range [0, 100] to handle out-of-bounds streaming events
       return float(np.clip(risk_score, 0.0, 100.0))
   ```